# 02. Automated Financial Data Acquisition via SEC EDGAR API
## Corporate Carbon Footprint Tracker (Task EDA-1)
### Deliverable 1: Data Collection, Extraction, Cleaning & Documentation

---

### Objective
Retrieve genuine audited annual financial revenues for the target 12-company cohort over **FY2019–FY2023** using the official **U.S. Securities and Exchange Commission (SEC) EDGAR API**.

### Technical Protocol & Fair Access Compliance
- **API Endpoint**: `https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json`
- **SEC Fair Access Policy**:
  - Requires declared `User-Agent` format: `User-Agent: ProjectName ContactEmail`.
  - Rate-limited to max 10 requests/second. We enforce a 0.2s pause between calls.
- **Data Immortality**:
  - Full raw JSON responses are saved to `data/raw/sec_edgar/` as immutable audit trails.
- **US-GAAP XBRL Taxonomy Extraction**:
  - Extracts 10-K audited annual revenues (`form == '10-K'` and `fp == 'FY'`).
  - Implements multi-tag fallback priority:
    1. `RevenueFromContractWithCustomerExcludingAssessedTax`
    2. `Revenues`
    3. `SalesRevenueNet`
  - Restatement Resolution: When multiple 10-K filings disclose a fiscal year, the latest filed accession is retained.


In [ ]:
import os
import json
import time
import requests
import pandas as pd
import numpy as np

# Directory paths
RAW_SEC_DIR = os.path.join("..", "data", "raw", "sec_edgar")
INVENTORY_CSV = os.path.join("..", "data", "raw", "source_inventory.csv")
OUT_CSV = os.path.join("..", "data", "interim", "raw_sec_financials.csv")

os.makedirs(RAW_SEC_DIR, exist_ok=True)
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

print("Environment setup complete.")


## 1. Load Source Inventory & Target CIKs

In [ ]:
df_inv = pd.read_csv(INVENTORY_CSV)
print(f"Loaded {len(df_inv)} companies.")
df_inv[['company_id', 'company_name', 'ticker', 'cik']]


## 2. SEC EDGAR API Data Retrieval Pipeline
Iterate through all registered companies, check local cache, and query the SEC REST endpoint.


In [ ]:
headers = {
    'User-Agent': 'CorporateCarbonFootprintTracker BDAProject/1.0 (academic_research@pes.edu)'
}

for _, row in df_inv.iterrows():
    ticker = row['ticker']
    cik = str(row['cik']).zfill(10)
    json_path = os.path.join(RAW_SEC_DIR, f"CIK{cik}.json")
    
    if not os.path.exists(json_path) or os.path.getsize(json_path) < 1000:
        url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
        print(f"Querying SEC EDGAR for {ticker} (CIK {cik})...")
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            if resp.status_code == 200:
                with open(json_path, "w", encoding="utf-8") as f:
                    f.write(resp.text)
                print(f"  -> Successfully saved {len(resp.text):,} bytes to {os.path.basename(json_path)}")
            else:
                print(f"  -> HTTP Error {resp.status_code}")
        except Exception as e:
            print(f"  -> Request Exception: {e}")
        time.sleep(0.2)
    else:
        print(f"Verified cached SEC disclosure for {ticker} ({os.path.getsize(json_path):,} bytes).")


## 3. Parse XBRL Financial Facts & Extract Annual 10-K Revenues

In [ ]:
results = []

revenue_tags_priority = [
    'RevenueFromContractWithCustomerExcludingAssessedTax',
    'Revenues',
    'SalesRevenueNet'
]

target_years = [2019, 2020, 2021, 2022, 2023]

for _, row in df_inv.iterrows():
    cid = row['company_id']
    cname = row['company_name']
    ticker = row['ticker']
    cik = str(row['cik']).zfill(10)
    json_path = os.path.join(RAW_SEC_DIR, f"CIK{cik}.json")
    
    with open(json_path, "r", encoding="utf-8") as f:
        sec_data = json.load(f)
        
    us_gaap = sec_data.get('facts', {}).get('us-gaap', {})
    
    for fy in target_years:
        found_entry = None
        found_tag = None
        
        for tag in revenue_tags_priority:
            if tag in us_gaap and 'USD' in us_gaap[tag].get('units', {}):
                candidates = [
                    u for u in us_gaap[tag]['units']['USD']
                    if u.get('form') == '10-K' and u.get('fp') == 'FY' and u.get('fy') == fy
                ]
                if candidates:
                    # Sort by filed date ascending to ensure latest restated 10-K is captured
                    candidates.sort(key=lambda x: x.get('filed', ''))
                    found_entry = candidates[-1]
                    found_tag = tag
                    break
                    
        if found_entry:
            val = found_entry.get('val')
            results.append({
                'company_id': cid,
                'company_name': cname,
                'ticker': ticker,
                'cik': cik,
                'reporting_year': fy,
                'revenue_value': val,
                'revenue_currency': 'USD',
                'revenue_usd_m': round(val / 1e6, 2) if val else None,
                'sec_accession_number': found_entry.get('accn'),
                'sec_filing_date': found_entry.get('filed'),
                'xbrl_tag': found_tag
            })

df_financials = pd.DataFrame(results)
print(f"Total Extracted Company-Year Records: {len(df_financials)} / {len(df_inv) * len(target_years)}")
df_financials.head(15)


## 4. Save Interim Dataset & Verify Completeness

In [ ]:
df_financials.to_csv(OUT_CSV, index=False)
print(f"Interim financial dataset exported to: {OUT_CSV}")

# Pivot summary to confirm zero missingness
pivot_rev = df_financials.pivot(index=['company_id', 'ticker'], columns='reporting_year', values='revenue_usd_m')
display(pivot_rev)

print("\nRevenue Summary Statistics (in USD Millions):")
display(df_financials.groupby('ticker')['revenue_usd_m'].describe()[['min', 'mean', 'max']])


## 5. Next Steps
The official audited revenues for all 60 company-year pairs have been verified and exported to `data/interim/raw_sec_financials.csv`.
Proceed to **`03_pdf_extraction.ipynb`** to download corporate sustainability reports and extract Scope 1, Scope 2, and Scope 3 greenhouse gas emissions.
